# 02 — Dữ liệu thị trường, vĩ mô và tài chính

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Nguồn dữ liệu dùng trong thí nghiệm

- VN-Index lấy giá đóng cửa DNSE và 49 quan sát bổ sung/hiệu chỉnh từ CafeF.
- Tỷ giá USD/VND và vàng ngày XAU/USD lấy từ bản xuất Investing.com.
- CPI năm và GDP năm lấy từ World Bank WDI, đủ 2005–2024.
- Bốn chỉ tiêu tài chính hợp nhất của FPT lấy từ snapshot báo cáo quý VCI, có chuỗi liên tục từ quý I/2013.

Ghép lùi theo ngày khả dụng. VN-Index, tỷ giá và vàng dùng từ ngày lịch kế tiếp. Báo cáo quý và CPI/GDP năm giả định trễ 45 và 180 ngày.

In [ ]:
from fetch_supplemental import run_from_cache, RAW_DIR, DEFAULT_OUTPUT, MODEL_SUPPLEMENTAL_COLUMNS
# Chuẩn hóa các bản nguồn đã lưu; bảng hiển thị tập trung vào đầu vào mô hình.
audit = run_from_cache(RAW_DIR, DEFAULT_OUTPUT)
print("Số phiên FPT:", audit["rows"], "| Mốc cuối:", audit["cutoff_date"])
supp = pd.read_csv(DEFAULT_OUTPUT)[["date", *MODEL_SUPPLEMENTAL_COLUMNS]]
display(supp.tail())


## Độ phủ trên lịch FPT

Vàng mới có 4.858 quan sát đến 30/09/2025. CPI/GDP mỗi nguồn có 20 năm. Sau khi loại hai ngày HOSE đóng cửa 23–24/01/2018, lịch FPT dùng để ghép có 4.669 phiên. Chuỗi chung đủ 42 biến liên tục từ 15/05/2013.

In [ ]:
display(supp.isna().sum().to_frame('Số ô thiếu'))

## Kiểm tra lịch giao dịch và các quý còn thiếu

VN-Index có quan sát cho mọi phiên FPT từ năm 2013 sau khi đối chiếu nguồn. Các quý thiếu 2007Q3–2012Q4 thuộc lịch sử trước phạm vi huấn luyện. Ô dưới đọc kiểm toán, gồm cả phần lịch sử dài hơn.

In [ ]:
from audit_data_quality import review
quality = review(ROOT)
print("Ngày FPT chưa có VN-Index từ 2013:", len(quality["calendar_comparison"]["fpt_dates_absent_from_vnindex_in_study"]))
print("Quý tài chính còn thiếu:", quality["financial_missing_quarters_inside_archive"])
coverage = pd.DataFrame(quality["aligned_model_feature_coverage"]).T
display(coverage[["first_valid_session", "last_valid_session", "missing_rows", "internal_missing_rows"]])
